# 05 The cascade as a state machine

The encoder decides, the conformal gate checks, and an LLM sees only the messages the gate refuses. Control flow comes from the **state**, never from model text:
the LLM's reply is validated against the closed route list in code, and a reply outside the list goes to a person.

**What is real and what is replayed.** The slow tier here is Claude Haiku. Its answers for the 200 test messages were produced once, in a Claude Code session, from
the route descriptions alone (it never saw the labels), and recorded in `data/llm/test_haiku_answers.jsonl`. The escalation node below looks the recorded answer up by message id.
No API call is made in this notebook, so **no LLM latency or price is measured here**; the notebook counts calls instead.

In [1]:
import json, hashlib, time, sqlite3
import numpy as np
from pathlib import Path
from typing import TypedDict, Optional, Literal
from scipy.special import softmax
from scipy.stats import binomtest
from sklearn.linear_model import LogisticRegression
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
import meridian as m

RESULTS = Path("data/results")
r01 = json.loads((RESULTS / "01_encoders_and_baseline.json").read_text())
r03 = json.loads((RESULTS / "03_calibration_and_conformal.json").read_text())
ENC, T, ALPHA = r03["encoder"], r03["T"], r03["alpha_chosen"]
Q = r03[f"conformal_alpha_{ALPHA}"]["q"]
print(f"encoder {ENC} | temperature {T:.2f} | conformal threshold q={Q:.3f} for alpha={ALPHA} | hardware: {m.hardware()}")

tr, te = m.load("train"), m.load("test")
_, ytr = m.texts_labels(tr); _, yte = m.texts_labels(te)
E = m.cached_embeddings(ENC)
clf = LogisticRegression(C=r01[ENC]["C"], max_iter=3000).fit(E["train"], ytr)
idx_of = {r["id"]: i for i, r in enumerate(te)}

idmap = json.loads(Path("data/llm/test_id_map.json").read_text())               # opaque id -> message id
llm_answers = {idmap[j["id"]]: j["route"] for j in map(json.loads, Path("data/llm/test_haiku_answers.jsonl").read_text().splitlines())}
print(f"recorded slow-tier answers: {len(llm_answers)}; outside the closed list: {sum(a not in m.ROUTES for a in llm_answers.values())}")

encoder bge-small | temperature 0.28 | conformal threshold q=0.849 for alpha=0.02 | hardware: Intel(R) Core(TM) i7-7567U CPU @ 3.50GHz, 4 logical cores, Darwin 22.6.0


recorded slow-tier answers: 200; outside the closed list: 0


## The state
One typed record travels through the graph. Every field is written by exactly one node.

In [2]:
class State(TypedDict, total=False):
    message_id: str
    text: str
    route_set: list[str]                         # conformal prediction set, set by `gate`
    decision: Literal["act", "escalate", "human"]
    route: Optional[str]                         # final route, set by act / escalate / review
    handled_by: Literal["encoder", "llm", "person"]
    status: str                                  # set by `execute`
    idem_key: str

DECISION_LOGIC_VERSION = "gate-v1:alpha=0.02"   # bump when the gate, the model or the thresholds change

## Nodes: classify and gate
`classify` is one encoder forward pass plus the head. `gate` turns the probabilities into a set and the set into a decision: one route acts, several escalate, none goes to a person.

In [3]:
loaded = m.load_encoder(ENC)
def classify(state: State) -> State:
    x = m.encode(ENC, [state["text"]], _loaded=loaded)
    p = softmax(clf.decision_function(x) / T, axis=1)[0]
    return {"route_set": [m.ROUTES[i] for i in np.where(p >= 1 - Q)[0]]}

def gate(state: State) -> State:
    s = state["route_set"]
    if len(s) == 1: return {"decision": "act", "route": s[0], "handled_by": "encoder"}
    return {"decision": "escalate" if len(s) > 1 else "human"}

/Users/ben/deterministic-ai-workshop/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## The escalation node: a closed reply, validated in code
The recorded reply must be one of the eight route names. Anything else, or no recorded reply, routes to a person. The type guarantees the format; whether the route is right is measured below.

In [4]:
LLM_CALLS = []     # one entry per slow-tier call, so the scorecard can count them

def escalate(state: State) -> State:
    LLM_CALLS.append(state["message_id"])
    reply = llm_answers.get(state["message_id"])
    if reply in m.ROUTES: return {"route": reply, "handled_by": "llm"}
    return {"decision": "human"}

def review(state: State) -> State:
    # Pauses the graph and persists its state. The caller resumes it with a person's decision.
    answer = interrupt({"message_id": state["message_id"], "text": state["text"], "candidates": state.get("route_set", [])})
    return {"route": answer, "handled_by": "person"}

## Idempotent execute
The side effect (here: writing the message to a queue) must happen at most once per message and decision logic. The key is a hash of the message id, the decision-logic version
and the action. Run the same message twice and the second run returns the stored result without writing.

In [5]:
db = sqlite3.connect(":memory:", check_same_thread=False)
db.execute("CREATE TABLE done (idem_key TEXT PRIMARY KEY, status TEXT)")
db.execute("CREATE TABLE queue_writes (message_id TEXT, route TEXT)")

def execute(state: State) -> State:
    action = f"queue:{state['route']}"
    key = hashlib.sha256(f"{state['message_id']}|{DECISION_LOGIC_VERSION}|{action}".encode()).hexdigest()[:16]
    row = db.execute("SELECT status FROM done WHERE idem_key=?", (key,)).fetchone()
    if row: return {"idem_key": key, "status": f"replayed:{row[0]}"}
    db.execute("INSERT INTO queue_writes VALUES (?,?)", (state["message_id"], state["route"]))
    db.execute("INSERT INTO done VALUES (?,?)", (key, "written"))
    return {"idem_key": key, "status": "written"}

## Wire the graph
Edges are chosen by `state["decision"]`, a field the code set, never by parsing model text.

In [6]:
def build(checkpointer=None):
    g = StateGraph(State)
    for name, fn in [("classify", classify), ("gate", gate), ("escalate", escalate), ("review", review), ("execute", execute)]:
        g.add_node(name, fn)
    g.add_edge(START, "classify"); g.add_edge("classify", "gate")
    g.add_conditional_edges("gate", lambda s: s["decision"], {"act": "execute", "escalate": "escalate", "human": "review"})
    g.add_conditional_edges("escalate", lambda s: s["decision"] if s.get("decision") == "human" else "act", {"act": "execute", "human": "review"})
    g.add_edge("review", "execute"); g.add_edge("execute", END)
    return g.compile(checkpointer=checkpointer)

app = build(MemorySaver())

### One message that acts, run twice

In [7]:
r = te[0]
cfg = {"configurable": {"thread_id": "demo-1"}}
out1 = app.invoke({"message_id": r["id"], "text": r["text"]}, cfg)
print("first run :", out1["decision"], "->", out1["route"], "|", out1["status"], "| key", out1["idem_key"])
out2 = app.invoke({"message_id": r["id"], "text": r["text"]}, {"configurable": {"thread_id": "demo-2"}})
print("second run:", out2["decision"], "->", out2["route"], "|", out2["status"], "| key", out2["idem_key"])
print("rows in queue_writes for this message:", db.execute("SELECT COUNT(*) FROM queue_writes WHERE message_id=?", (r["id"],)).fetchone()[0])

first run : act -> baggage_claim | written | key e383172571dc608c


second run: act -> baggage_claim | replayed:written | key e383172571dc608c
rows in queue_writes for this message: 1


### One message that pauses for a person
At alpha = 0.02 no test message produces an empty set, so the gate never sends one straight to a person. The person path is still reached when the slow tier fails:
a timeout, a malformed reply, a reply outside the route list. Here we simulate that by removing one recorded reply, run the message until the graph pauses, then resume it with a decision.

In [8]:
def set_size(row):
    x = m.encode(ENC, [row["text"]], _loaded=loaded); p = softmax(clf.decision_function(x) / T, axis=1)[0]
    return int((p >= 1 - Q).sum())
sizes = [set_size(r) for r in te]
print("set-size histogram on the test split:", {s: sizes.count(s) for s in sorted(set(sizes))})
pick = next(r for r, s in zip(te, sizes) if s > 1)
saved = llm_answers.pop(pick["id"])                      # simulate: the slow tier returned nothing usable
cfg = {"configurable": {"thread_id": "demo-human"}}
paused = app.invoke({"message_id": pick["id"], "text": pick["text"]}, cfg)
print("candidates the gate kept:", paused["route_set"])
print("paused for review:", "__interrupt__" in paused, "| rows written so far:", db.execute("SELECT COUNT(*) FROM queue_writes WHERE message_id=?", (pick["id"],)).fetchone()[0])
done = app.invoke(Command(resume=paused["route_set"][0]), cfg)
print("resumed ->", done["route"], "| handled_by", done["handled_by"], "|", done["status"])
llm_answers[pick["id"]] = saved                          # restore the recorded reply

set-size histogram on the test split: {1: 183, 2: 15, 3: 2}
candidates the gate kept: ['cancellation_request', 'rebooking_request', 'refund_claim']
paused for review: True | rows written so far: 0
resumed -> cancellation_request | handled_by person | written


## Run the whole test split

In [9]:
LLM_CALLS.clear()
rows_out, t0 = [], time.perf_counter()
ids_human = set()
for r in te:
    cfg = {"configurable": {"thread_id": f"run-{r['id']}"}}
    o = app.invoke({"message_id": r["id"], "text": r["text"]}, cfg)
    if "__interrupt__" in o:
        rows_out.append(dict(id=r["id"], route=None, by="person-queue", decision=o.get("decision", "human"))); continue
    rows_out.append(dict(id=r["id"], route=o["route"], by=o["handled_by"], decision=o["decision"]))
wall = time.perf_counter() - t0
print(f"{len(te)} messages through the graph in {wall:.1f} s on this CPU ({wall / len(te) * 1000:.0f} ms per message, includes recorded-reply lookups)")

200 messages through the graph in 16.4 s on this CPU (82 ms per message, includes recorded-reply lookups)


## The scorecard against an LLM-only router
Accuracy is measured on the messages the cascade answered itself (encoder or LLM). Messages sent to a person are reported as a share, not scored.
The LLM-only router is the same recorded Haiku reply for all 200 messages. Intervals: Wilson (assumes independent items) and a cluster bootstrap over scenarios.

In [10]:
y = {r["id"]: yte[i] for i, r in enumerate(te)}
cl = {r["id"]: f"{r['label']}:{r['scenario']}" for r in te}
answered = [o for o in rows_out if o["route"] is not None]
ok_c = np.array([m.ROUTES.index(o["route"]) == y[o["id"]] for o in answered])
cl_c = np.array([cl[o["id"]] for o in answered])
n_enc = sum(o["by"] == "encoder" for o in rows_out); n_llm = sum(o["by"] == "llm" for o in rows_out); n_per = sum(o["route"] is None or o["by"] == "person" for o in rows_out)
print(f"handled by encoder {n_enc} ({n_enc/len(te):.1%}) | escalated to LLM {n_llm} ({n_llm/len(te):.1%}) | sent to a person {n_per} ({n_per/len(te):.1%})")
lo, hi = m.wilson(int(ok_c.sum()), len(ok_c)); clo, chi = m.cluster_bootstrap_ci(ok_c.astype(float), cl_c)
print(f"cascade accuracy on answered ({len(ok_c)}): {ok_c.mean():.3f}  Wilson [{lo:.3f}, {hi:.3f}]  cluster [{clo:.3f}, {chi:.3f}]")
for who in ("encoder", "llm"):
    s = np.array([m.ROUTES.index(o["route"]) == y[o["id"]] for o in answered if o["by"] == who])
    if len(s): print(f"   of which {who:8s} n={len(s):3d} accuracy {s.mean():.3f}  Wilson [{m.wilson(int(s.sum()), len(s))[0]:.3f}, {m.wilson(int(s.sum()), len(s))[1]:.3f}]")

llm_ok = np.array([m.ROUTES.index(llm_answers[r["id"]]) == yte[i] if llm_answers.get(r["id"]) in m.ROUTES else False for i, r in enumerate(te)])
cl_all = np.array([cl[r["id"]] for r in te])
lo2, hi2 = m.wilson(int(llm_ok.sum()), len(llm_ok)); clo2, chi2 = m.cluster_bootstrap_ci(llm_ok.astype(float), cl_all)
print(f"LLM-only accuracy on all {len(te)}: {llm_ok.mean():.3f}  Wilson [{lo2:.3f}, {hi2:.3f}]  cluster [{clo2:.3f}, {chi2:.3f}]")
print(f"slow-tier calls per 1,000 messages: LLM-only 1000 | cascade {len(LLM_CALLS) / len(te) * 1000:.0f}")

handled by encoder 183 (91.5%) | escalated to LLM 17 (8.5%) | sent to a person 0 (0.0%)


cascade accuracy on answered (200): 0.975  Wilson [0.943, 0.989]  cluster [0.935, 1.000]
   of which encoder  n=183 accuracy 0.984  Wilson [0.953, 0.994]
   of which llm      n= 17 accuracy 0.882  Wilson [0.657, 0.967]


LLM-only accuracy on all 200: 0.950  Wilson [0.910, 0.973]  cluster [0.875, 1.000]
slow-tier calls per 1,000 messages: LLM-only 1000 | cascade 85


### Paired comparison on the messages the cascade answered
Exact McNemar test: among messages where exactly one system is right, is one system right more often than chance? Discordant counts, then the two-sided exact p-value.

In [11]:
llm_on_answered = np.array([llm_ok[idx_of[o["id"]]] for o in answered])
b = int(((ok_c) & (~llm_on_answered)).sum()); c = int(((~ok_c) & (llm_on_answered)).sum())
print(f"cascade right, LLM-only wrong: {b} | cascade wrong, LLM-only right: {c} | both right: {int((ok_c & llm_on_answered).sum())} | both wrong: {int((~ok_c & ~llm_on_answered).sum())}")
p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
print(f"exact McNemar two-sided p = {p:.3f}   (messages are variants of 40 scenarios, so treat this p-value as optimistic)")

res = dict(n=len(te), encoder=n_enc, llm=n_llm, person=n_per, cascade_acc=float(ok_c.mean()), cascade_wilson=[lo, hi], cascade_cluster=[clo, chi],
           llm_only_acc=float(llm_ok.mean()), llm_only_wilson=[lo2, hi2], llm_only_cluster=[clo2, chi2],
           llm_calls_per_1000_cascade=len(LLM_CALLS) / len(te) * 1000, mcnemar=dict(b=b, c=c, p=float(p)),
           ms_per_message_graph=wall / len(te) * 1000, hardware=m.hardware(), decision_logic_version=DECISION_LOGIC_VERSION)
(RESULTS / "05_langgraph_cascade.json").write_text(json.dumps(res, indent=2, default=float))

cascade right, LLM-only wrong: 8 | cascade wrong, LLM-only right: 3 | both right: 187 | both wrong: 2
exact McNemar two-sided p = 0.227   (messages are variants of 40 scenarios, so treat this p-value as optimistic)


638